# Variance Risk Premium: Bull Put Spread Backtest on SPY

The VIX tends to overstate how much SPY actually moves. That gap between implied and realized volatility is the variance risk premium (VRP), and it has been positive on about 85% of trading days since 2010. Investors overpay for downside protection, and that overpayment goes to the options seller over time.

This notebook quantifies the VRP using 16 years of real SPY and VIX data and backtests a monthly bull put credit spread that uses the VRP level as an entry filter. The main question: does only trading when the premium is elevated actually improve risk-adjusted returns compared to just trading every month?

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings('ignore')
%matplotlib inline

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Strategy parameters
SPREAD_WIDTH  = 0.02   # short strike at -2%, long strike at -4%
PREMIUM_RATIO = 0.35   # net credit as fraction of spread width
VRP_THRESHOLD = 3.0    # minimum VRP (%) to enter
CAPITAL_ALLOC = 0.20   # portfolio fraction per trade
RV_WINDOW     = 21     # rolling window for realized vol (trading days)
START, END    = '2010-01-01', '2025-12-31'

STRESS_EVENTS = [
    ('2011 Credit\nDowngrade', '2011-08-08', '2011-07-01', '2011-09-30'),
    ('2015 Flash\nCrash',      '2015-08-24', '2015-07-15', '2015-10-15'),
    ('2018\nVolmageddon',      '2018-02-05', '2018-01-01', '2018-03-31'),
    ('2020 COVID\nCrash',      '2020-03-16', '2020-02-01', '2020-05-31'),
    ('2022 Fed\nRate Shock',   '2022-06-16', '2022-04-01', '2022-09-30'),
]

C_IV, C_RV   = '#2C7BB6', '#D7191C'
C_POS, C_NEG = '#4DAC26', '#D01C8B'
C_GREY       = '#AAAAAA'

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

## 1. Data

Daily SPY prices and CBOE VIX from Yahoo Finance. The VIX is already in annualized percentage terms, so I can compare it directly to realized vol without any unit conversion.

Realized vol is the rolling 21-day standard deviation of log returns, annualized:

```
RV_t = std(log returns, 21-day window) * sqrt(252) * 100
```

21 trading days is about one calendar month, which lines up with the VIX's 30-day horizon. VRP each day is:

```
VRP_t = VIX_t - RV_t
```

Positive means options are priced rich relative to what vol has been doing lately. Negative means realized vol spiked past what was priced in, which is what happens during crashes.

In [ ]:
spy = yf.download('SPY',  start=START, end=END, auto_adjust=True, progress=False)
vix = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)

df = pd.DataFrame({'SPY': spy['Close'].squeeze(), 'VIX': vix['Close'].squeeze()}).dropna()
df['ret'] = np.log(df['SPY'] / df['SPY'].shift(1))
df['RV']  = df['ret'].rolling(RV_WINDOW).std() * np.sqrt(252) * 100
df['IV']  = df['VIX']
df['VRP'] = df['IV'] - df['RV']
df = df.dropna()

n_days     = len(df)
n_years    = (df.index[-1] - df.index[0]).days / 365.25
date_start = df.index[0].strftime('%Y-%m-%d')
date_end   = df.index[-1].strftime('%Y-%m-%d')
vrp_mean   = df['VRP'].mean()
vrp_median = df['VRP'].median()
vrp_std    = df['VRP'].std()
iv_gt_rv   = (df['VRP'] > 0).mean() * 100
iv_mean    = df['IV'].mean()
rv_mean    = df['RV'].mean()

print(f'{n_days} trading days  ({date_start} to {date_end})')
print(f'Mean VRP: {vrp_mean:.2f}%   Median: {vrp_median:.2f}%   Std: {vrp_std:.2f}%')
print(f'IV > RV on {iv_gt_rv:.1f}% of days')
df[['IV', 'RV', 'VRP']].describe().round(2)

## 2. Backtest

Short put at 2% OTM, long put at 4% OTM, held to expiration. We collect 35% of the spread width as premium and risk the remaining 65%.

Payoff at expiration:
- SPY falls less than 2%: keep the full 35%
- SPY falls between 2% and 4%: partial loss based on how far through the spread it moved
- SPY falls more than 4%: max loss of 65%

20% of the portfolio goes into each trade, compounded monthly. Two versions run in parallel: the filtered strategy (only enter when VRP > 3%) and an always-on baseline that trades every month regardless.

In [ ]:
def spread_pnl(entry: float, exit_: float, premium: float) -> float:
    """P&L as a fraction of spread width for a bull put credit spread."""
    short_strike = entry * (1 - SPREAD_WIDTH)
    long_strike  = entry * (1 - 2 * SPREAD_WIDTH)
    if exit_ >= short_strike:
        return premium
    elif exit_ >= long_strike:
        return premium - (short_strike - exit_) / (entry * SPREAD_WIDTH)
    else:
        return premium - 1.0

month_ends = [d for d in df.resample('BME').last().index if d in df.index]

trades, ao_trades          = [], []
portfolio, always_on_pf    = 1.0, 1.0
portfolio_ts, always_on_ts = [], []

for i, entry_date in enumerate(month_ends[:-1]):
    exit_date = month_ends[i + 1]
    if exit_date not in df.index:
        continue

    entry_spy = float(df.loc[entry_date, 'SPY'])
    exit_spy  = float(df.loc[exit_date,  'SPY'])
    vrp_entry = float(df.loc[entry_date, 'VRP'])
    pnl       = spread_pnl(entry_spy, exit_spy, PREMIUM_RATIO)

    always_on_pf *= (1 + pnl * CAPITAL_ALLOC)
    always_on_ts.append({'date': exit_date, 'portfolio': always_on_pf})
    ao_trades.append({'pnl_pct': pnl * 100, 'win': pnl > 0})

    if vrp_entry >= VRP_THRESHOLD:
        portfolio *= (1 + pnl * CAPITAL_ALLOC)
        trades.append({
            'entry_date': entry_date, 'exit_date': exit_date,
            'entry_spy': entry_spy,   'exit_spy': exit_spy,
            'vrp_entry': vrp_entry,   'spy_ret': (exit_spy / entry_spy) - 1,
            'pnl_pct': pnl * 100,     'win': pnl > 0,
        })
    portfolio_ts.append({'date': exit_date, 'portfolio': portfolio})

trades_df    = pd.DataFrame(trades)
ao_trades_df = pd.DataFrame(ao_trades)
pf_ts        = pd.DataFrame(portfolio_ts).set_index('date')
ao_ts        = pd.DataFrame(always_on_ts).set_index('date')

print(f'Filtered trades: {len(trades_df)}  |  Always-on trades: {len(ao_trades_df)}')

In [ ]:
n_trades     = len(trades_df)
win_rate     = trades_df['win'].mean() * 100
avg_pnl      = trades_df['pnl_pct'].mean()
avg_pnl_wins = trades_df.loc[trades_df['win'],  'pnl_pct'].mean()
avg_pnl_loss = trades_df.loc[~trades_df['win'], 'pnl_pct'].mean()
cum_ret_filt = (portfolio - 1) * 100
cum_ret_ao   = (always_on_pf - 1) * 100
ann_ret_filt = ((1 + cum_ret_filt / 100) ** (1 / n_years) - 1) * 100
ann_ret_ao   = ((1 + cum_ret_ao   / 100) ** (1 / n_years) - 1) * 100
ao_win_rate  = ao_trades_df['win'].mean() * 100

pf_ts['peak'] = pf_ts['portfolio'].cummax()
pf_ts['dd']   = (pf_ts['portfolio'] - pf_ts['peak']) / pf_ts['peak'] * 100
max_dd        = pf_ts['dd'].min()

ao_ts['peak'] = ao_ts['portfolio'].cummax()
ao_ts['dd']   = (ao_ts['portfolio'] - ao_ts['peak']) / ao_ts['peak'] * 100
ao_max_dd     = ao_ts['dd'].min()

trade_rets    = trades_df['pnl_pct'] / 100 * CAPITAL_ALLOC
sharpe_annual = (trade_rets.mean() / trade_rets.std()) * np.sqrt(12)

import pandas as pd
summary = pd.DataFrame({
    'Metric': ['Trades', 'Win Rate', 'Avg P&L', 'Ann. Return', 'Max Drawdown', 'Sharpe'],
    'Filtered': [n_trades, f'{win_rate:.1f}%', f'{avg_pnl:.2f}%',
                 f'{ann_ret_filt:.1f}%', f'{max_dd:.2f}%', f'{sharpe_annual:.2f}'],
    'Always-On': [len(ao_trades_df), f'{ao_win_rate:.1f}%', '--',
                  f'{ann_ret_ao:.1f}%', f'{ao_max_dd:.2f}%', '--'],
}).set_index('Metric')
summary

## 3. Figures

### Figure 1: IV vs. RV and Daily VRP

Top panel is the VIX and 21-day realized vol over the full sample. Blue shading is when IV is above RV, which is the normal state. Red shading flips that and marks stress periods where realized vol overtook implied vol. The five major events are labeled at their VIX peak. Bottom panel is the daily VRP with the 3% entry threshold.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8),
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df.index, df['IV'], color=C_IV, lw=1.2, label='IV (VIX)', zorder=3)
ax1.plot(df.index, df['RV'], color=C_RV, lw=1.2, label='21-day RV', zorder=3)
ax1.fill_between(df.index, df['IV'], df['RV'],
                 where=(df['IV'] >= df['RV']), alpha=0.15, color=C_IV, label='IV > RV')
ax1.fill_between(df.index, df['IV'], df['RV'],
                 where=(df['IV'] <  df['RV']), alpha=0.20, color=C_RV, label='RV > IV')

for label, peak_date, _, _ in STRESS_EVENTS:
    try:
        ts  = pd.Timestamp(peak_date)
        idx = ts if ts in df.index else df.index[df.index.searchsorted(ts)]
        ax1.annotate(label, xy=(ts, df.loc[idx, 'IV']),
                     xytext=(0, 18), textcoords='offset points',
                     fontsize=7.5, ha='center',
                     arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8))
    except Exception:
        pass

ax1.set_ylabel('Volatility (%)')
ax1.set_title('Implied vs. Realized Volatility and the Variance Risk Premium (2010-2025)')
ax1.legend(loc='upper right', fontsize=8)

colors_bar = [C_IV if v > 0 else C_RV for v in df['VRP']]
ax2.bar(df.index, df['VRP'], color=colors_bar, width=2, alpha=0.7)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(VRP_THRESHOLD, color=C_GREY, lw=1, ls='--',
            label=f'Entry threshold ({VRP_THRESHOLD}%)')
ax2.set_ylabel('VRP (%)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig1_vrp_time.png', dpi=150, bbox_inches='tight')
plt.show()

### Figure 2: VRP Distribution and Quintile Forward Returns

Left panel is the distribution of daily VRP values. Most days cluster between 0 and 10%, with a left tail from crash periods where realized vol ran past implied vol.

Right panel breaks SPY's 21-day forward return by VRP quintile. Q5 (highest VRP) has the best forward returns. Q1 (negative VRP) also looks strong because those are post-crash environments where the market tends to bounce. Q4 being weaker than Q2 and Q3 is a bit odd, but the differences across the middle quintiles are small.

In [ ]:
df['fwd_ret_21'] = df['SPY'].pct_change(RV_WINDOW).shift(-RV_WINDOW) * 100
df_valid = df.dropna(subset=['fwd_ret_21', 'VRP']).copy()
df_valid['quintile'] = pd.qcut(
    df_valid['VRP'], 5,
    labels=['Q1\n(Low)', 'Q2', 'Q3', 'Q4', 'Q5\n(High)']
)
quintile_means = df_valid.groupby('quintile', observed=True)['fwd_ret_21'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(df['VRP'], bins=80, color=C_IV, alpha=0.75, edgecolor='none')
ax1.axvline(vrp_mean,   color='black', lw=1.5, ls='-',  label=f'Mean ({vrp_mean:.2f}%)')
ax1.axvline(vrp_median, color=C_RV,   lw=1.5, ls='--', label=f'Median ({vrp_median:.2f}%)')
ax1.axvline(0,          color='grey', lw=1,   ls=':')
ax1.set_xlabel('VRP (%)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Daily VRP')
ax1.legend()

bar_colors = [C_RV if v < 0 else C_IV for v in quintile_means.values]
ax2.bar(quintile_means.index.astype(str), quintile_means.values, color=bar_colors, alpha=0.8)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('VRP Quintile')
ax2.set_ylabel('Avg 21-day Forward SPY Return (%)')
ax2.set_title('VRP Quintile vs. Forward SPY Returns')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig2_vrp_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### Figure 3: Backtest Results

Top-left is cumulative P&L on log scale. Log scale is necessary because the always-on strategy ends at around 120,000% and filtered at about 15,700%. On a linear axis the filtered line would look nearly flat.

The scatter (bottom-left) is the most useful panel. Green is wins, pink is losses. Losses show up across the full range of VRP values at entry, so entering with a high VRP does not guarantee a win. The filter shifts the odds, it does not remove the tail.

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

ax1.plot(pf_ts.index, pf_ts['portfolio'], color=C_IV,  lw=2,   label='VRP-filtered')
ax1.plot(ao_ts.index, ao_ts['portfolio'], color=C_GREY, lw=1.5, ls='--', label='Always-on')
ax1.axhline(1, color='black', lw=0.5)
ax1.set_yscale('log')
ax1.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda y, _: f'{(y - 1) * 100:.0f}%')
)
ax1.set_ylabel('Cumulative Return (log scale)')
ax1.set_title('Cumulative Portfolio P&L')
ax1.legend()

bar_cols = [C_POS if w else C_NEG for w in trades_df['win']]
ax2.bar(range(n_trades), trades_df['pnl_pct'], color=bar_cols, alpha=0.8)
ax2.axhline(0, color='black', lw=0.8)
ax2.legend(handles=[mpatches.Patch(color=C_POS, label='Win'),
                    mpatches.Patch(color=C_NEG, label='Loss')])
ax2.set_xlabel('Trade #')
ax2.set_ylabel('P&L (% of spread width)')
ax2.set_title('Monthly Trade P&L')

short_ret      = -SPREAD_WIDTH * 100
long_ret       = -2 * SPREAD_WIDTH * 100
scatter_colors = [C_POS if w else C_NEG for w in trades_df['win']]
ax3.scatter(trades_df['vrp_entry'], trades_df['spy_ret'] * 100,
            c=scatter_colors, alpha=0.7, s=40)
ax3.axhline(short_ret, color='orange', lw=1.5, ls='--', label=f'Short strike ({short_ret:.0f}%)')
ax3.axhline(long_ret,  color=C_RV,    lw=1.5, ls=':',  label=f'Long strike ({long_ret:.0f}%)')
ax3.set_xlabel('VRP at Entry (%)')
ax3.set_ylabel('SPY Return over Trade Period (%)')
ax3.set_title('VRP at Entry vs. SPY Return')
ax3.legend(fontsize=8)

ax4.fill_between(pf_ts.index, pf_ts['dd'], 0, alpha=0.5, color=C_RV)
ax4.plot(pf_ts.index, pf_ts['dd'], color=C_RV, lw=1)
ax4.set_ylabel('Drawdown (%)')
ax4.set_xlabel('Date')
ax4.set_title('Strategy Drawdown Profile')

fig.suptitle('Backtest Results: VRP-Filtered Bull Put Spread Strategy', fontsize=13, y=1.01)
fig.savefig(OUTPUT_DIR / 'fig3_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

### Figure 4: Tail Risk

Left panel compares average VRP before vs during each major stress event. In four of the five cases the VRP went negative during the event because realized vol outran what options had priced in. COVID is the most extreme, averaging around -12% during the event window. The 2022 rate shock is the outlier: the Fed's tightening was gradual and widely expected, so realized vol never really blew out and the VRP stayed positive.

Right panel shows win rate at different entry thresholds. It starts around 86% at a threshold of 0% and stays roughly flat through 3-4%. Above 7% it gets noisy because sample sizes shrink. The filter is mostly cutting out low-premium months rather than finding particularly high-quality setups.

In [ ]:
event_labels, vrp_before, vrp_during = [], [], []
for label, peak_date, window_start, window_end in STRESS_EVENTS:
    ws  = pd.Timestamp(window_start)
    we  = pd.Timestamp(window_end)
    pk  = pd.Timestamp(peak_date)
    event_labels.append(label)
    vrp_before.append(df[(df.index >= ws) & (df.index <  pk)]['VRP'].mean())
    vrp_during.append(df[(df.index >= pk) & (df.index <= we)]['VRP'].mean())

thresholds   = np.arange(0, 12.5, 0.5)
win_rates_t  = []
trade_counts = []
for thr in thresholds:
    sub = trades_df[trades_df['vrp_entry'] >= thr]
    win_rates_t.append(sub['win'].mean() * 100 if len(sub) else float('nan'))
    trade_counts.append(len(sub))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

x = np.arange(len(event_labels))
w = 0.35
ax1.bar(x - w/2, vrp_before, w, label='Before event', color=C_IV, alpha=0.8)
ax1.bar(x + w/2, vrp_during, w, label='During event', color=C_RV, alpha=0.8)
ax1.axhline(0, color='black', lw=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(event_labels, fontsize=8)
ax1.set_ylabel('Average VRP (%)')
ax1.set_title('VRP Before vs. During Major Stress Events')
ax1.legend()

ax2_r = ax2.twinx()
ax2.plot(thresholds, win_rates_t, color=C_IV, lw=2, marker='o', ms=4, label='Win rate')
ax2.axvline(VRP_THRESHOLD, color='orange', lw=1.5, ls='--',
            label=f'Chosen threshold ({VRP_THRESHOLD}%)')
ax2.set_xlabel('VRP Entry Threshold (%)')
ax2.set_ylabel('Win Rate (%)', color=C_IV)
ax2.tick_params(axis='y', labelcolor=C_IV)
ax2.set_title('Win Rate vs. Entry Threshold')
ax2_r.bar(thresholds, trade_counts, width=0.4, alpha=0.25, color=C_GREY, label='Trade count')
ax2_r.set_ylabel('Number of Trades', color=C_GREY)
ax2_r.tick_params(axis='y', labelcolor=C_GREY)
lines1, l1 = ax2.get_legend_handles_labels()
lines2, l2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, l1 + l2, fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig4_tail_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Summary

The VRP averaged 3.67% (median 4.10%) and was positive on 85.3% of trading days. That base rate is what makes this trade viable.

The filter does not move the win rate much. Both versions win around 85-86% of months. The difference shows up in the drawdown: filtered max drawdown is -24.3% vs -40.7% for always-on. You give up return to get that (37.5% vs 56.2% annualized).

A few things worth keeping in mind:
- The Sharpe of 2.47 uses sqrt(12) annualization, which overstates it since only about 7-8 months per year are active. Adjusted for actual trade frequency it is closer to 2.0.
- 2010-2025 was a strong period for US equities. These numbers are probably an upper bound on what live performance would look like.
- No intra-month stop is modeled. In practice you would close early if the position moved against you mid-month, which would affect the loss distribution.

In [ ]:
vix_peaks = {}
for label, peak_date, window_start, window_end in STRESS_EVENTS:
    sub = df[(df.index >= pd.Timestamp(window_start)) &
             (df.index <= pd.Timestamp(window_end))]
    if len(sub):
        vix_peaks[label.replace('\n', ' ')] = {
            'peak_vix':  round(float(sub['IV'].max()), 1),
            'peak_date': sub['IV'].idxmax().strftime('%Y-%m-%d'),
        }

quintile_tbl = {
    str(q): {
        'vrp_mean': round(float(df_valid[df_valid['quintile'] == q]['VRP'].mean()), 2),
        'fwd_ret':  round(float(df_valid[df_valid['quintile'] == q]['fwd_ret_21'].mean()), 3),
    }
    for q in df_valid['quintile'].cat.categories
}

output = {
    'date_start': date_start, 'date_end': date_end,
    'n_days': int(n_days), 'n_years': round(n_years, 1),
    'vrp_mean': round(vrp_mean, 2), 'vrp_median': round(vrp_median, 2),
    'vrp_std': round(vrp_std, 2), 'iv_gt_rv_pct': round(iv_gt_rv, 1),
    'iv_mean': round(iv_mean, 2), 'rv_mean': round(rv_mean, 2),
    'n_trades': int(n_trades), 'win_rate': round(win_rate, 1),
    'avg_pnl': round(avg_pnl, 2), 'avg_pnl_wins': round(avg_pnl_wins, 2),
    'avg_pnl_losses': round(avg_pnl_loss, 2),
    'cum_ret_filtered': round(cum_ret_filt, 2),
    'cum_ret_always_on': round(cum_ret_ao, 2),
    'ann_ret_filtered': round(ann_ret_filt, 1),
    'ann_ret_always_on': round(ann_ret_ao, 1),
    'max_drawdown': round(max_dd, 2), 'ao_win_rate': round(ao_win_rate, 1),
    'ao_max_dd': round(ao_max_dd, 2), 'n_ao_trades': int(len(ao_trades_df)),
    'sharpe_annual': round(sharpe_annual, 2),
    'vrp_threshold': VRP_THRESHOLD, 'spread_width_pct': SPREAD_WIDTH * 100,
    'premium_ratio_pct': PREMIUM_RATIO * 100, 'capital_alloc_pct': CAPITAL_ALLOC * 100,
    'rv_window': RV_WINDOW, 'vix_peaks': vix_peaks, 'quintile_fwd_ret': quintile_tbl,
}

with open('stats.json', 'w') as f:
    json.dump(output, f, indent=2)

print('stats.json saved.')
print(f'Figures saved to {OUTPUT_DIR}/')